# Prognoza zapotrzebowania na ciepło — wersja uporządkowana

Notebook przygotowuje model **day-ahead**: prognoza wykonywana dzień wcześniej o 08:00 na cały kolejny dzień w interwale 15-minutowym.

Najważniejsze założenia:
- surowy profil Helsinek jest skalowany **raz** do mocy cieplnej 5 × Jenbacher J920,
- zapisujemy wspólny plik `helsinki_heat_scaled_15min.csv`, z którego korzysta później zarówno trening, jak i `Serce_modelu`,
- target modelu to `heat_MW`, czyli moc cieplna, a energia 15-minutowa jest liczona jako `heat_MW * 0.25`,
- model nie używa krótkich lagów ciepła, których nie znalibyśmy w chwili prognozy D-1 08:00.

In [ ]:
# Opcjonalnie uruchom tylko wtedy, gdy brakuje bibliotek w środowisku
# !pip install pandas numpy matplotlib scikit-learn requests xgboost joblib

In [ ]:
import os
import joblib
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Parametry i ścieżki

In [ ]:
# =========================
# PARAMETRY GŁÓWNE
# =========================

RAW_HELSINKI_PATH = "helen_2015_2024_v2.csv"
SCALED_HELSINKI_15MIN_PATH = "helsinki_heat_scaled_15min.csv"
MODELING_DATA_PATH = "helsinki_heat_weather_scaled_15min.csv"
HEAT_MODEL_PATH = "heat_demand_forecast.pkl"

# Lokalizacja Helsinek
LAT = 60.1699
LON = 24.9384

# Parametry źródła ciepła: 5 x Jenbacher J920
N_UNITS = 5
THERMAL_POWER_ONE_UNIT_MW = 10.3
ELECTRIC_POWER_ONE_UNIT_MW = 10.606

THERMAL_POWER_TOTAL_MW = N_UNITS * THERMAL_POWER_ONE_UNIT_MW
ELECTRIC_POWER_TOTAL_MW = N_UNITS * ELECTRIC_POWER_ONE_UNIT_MW

# Parametry modelu ciepła
BASE_TEMP = 17
ISSUE_HOUR = 8
RANDOM_STATE = 42

print("Moc cieplna łączna [MWth]:", THERMAL_POWER_TOTAL_MW)
print("Moc elektryczna łączna [MWel]:", ELECTRIC_POWER_TOTAL_MW)
print("Godzina wykonania prognozy day-ahead:", ISSUE_HOUR)

## 2. Funkcje pomocnicze

In [ ]:
def download_open_meteo_weather(lat, lon, start_date, end_date):
    """
    Pobiera historyczne dane pogodowe Open-Meteo dla podanego zakresu dat.
    Zwraca dane godzinowe w UTC z kolumną datetime.
    """
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": str(start_date),
        "end_date": str(end_date),
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m",
            "cloud_cover",
            "shortwave_radiation",
            "precipitation",
        ],
        "timezone": "UTC",
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()["hourly"]
    weather = pd.DataFrame(data)
    weather["time"] = pd.to_datetime(weather["time"])
    weather = weather.rename(columns={"time": "datetime"})

    return weather


def resample_weather_to_15min_grid(weather_hourly, target_datetimes):
    """
    Resampluje pogodę godzinową do siatki 15-minutowej zgodnej z profilem ciepła.
    Końcówki dopełniamy ffill/bfill, żeby nie gubić 23:15, 23:30, 23:45.
    """
    target_index = pd.DatetimeIndex(target_datetimes).sort_values()

    weather_15min = (
        weather_hourly
        .sort_values("datetime")
        .set_index("datetime")
        .resample("15min")
        .interpolate(method="time")
        .reindex(target_index)
        .interpolate(method="time")
        .ffill()
        .bfill()
        .rename_axis("datetime")
        .reset_index()
    )

    return weather_15min


def calculate_metrics(y_true, y_pred):
    """
    Zwraca podstawowe metryki jakości prognozy.

    y_true może mieć oryginalny indeks dataframe'u,
    a y_pred zwykle ma indeks 0..n.
    Dlatego konwertujemy oba obiekty do numpy arrays.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    non_zero = y_true != 0

    if non_zero.sum() == 0:
        mape = np.nan
    else:
        mape = np.mean(
            np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])
        ) * 100

    return {
        "MAE_MW": mae,
        "RMSE_MW": rmse,
        "MAPE_%": mape,
        "R2": r2,
    }

In [ ]:
def prepare_heat_day_ahead_features(df_heat_15min, issue_hour=8, base_temp=17):
    """
    Buduje cechy dla modelu day-ahead.

    Założenie operacyjne:
    - prognoza wykonywana jest D-1 o 08:00,
    - prognozujemy cały dzień D: 00:00–23:45,
    - znamy historię ciepła tylko do momentu issue_time,
    - znamy/scenariuszujemy pogodę dla dnia D.

    Dlatego nie używamy krótkich lagów ciepła typu 1h, 6h, 24h.
    Bezpieczne dla całego dnia D są m.in. heat_lag_48h i heat_lag_168h.
    """
    df = df_heat_15min.copy()
    df = df.sort_values("datetime").reset_index(drop=True)

    required_cols = [
        "datetime",
        "heat_MW",
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "cloud_cover",
        "shortwave_radiation",
        "precipitation",
    ]

    missing_required = [col for col in required_cols if col not in df.columns]
    if missing_required:
        raise ValueError(f"Brakuje wymaganych kolumn: {missing_required}")

    # Cechy kalendarzowe
    df["year"] = df["datetime"].dt.year
    df["month"] = df["datetime"].dt.month
    df["day"] = df["datetime"].dt.day
    df["hour"] = df["datetime"].dt.hour
    df["minute"] = df["datetime"].dt.minute
    df["dayofweek"] = df["datetime"].dt.dayofweek
    df["dayofyear"] = df["datetime"].dt.dayofyear
    df["weekofyear"] = df["datetime"].dt.isocalendar().week.astype(int)
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

    df["hour_fraction"] = df["hour"] + df["minute"] / 60

    df["hour_sin"] = np.sin(2 * np.pi * df["hour_fraction"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour_fraction"] / 24)

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    df["dayofyear_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365)
    df["dayofyear_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365)

    # Cechy pogodowe pochodne
    df["heating_degree"] = np.maximum(0, base_temp - df["temperature_2m"])
    df["cooling_degree"] = np.maximum(0, df["temperature_2m"] - base_temp)

    # Moment wydania prognozy dla każdego punktu czasu
    df["issue_time"] = (
        df["datetime"].dt.normalize()
        - pd.Timedelta(days=1)
        + pd.Timedelta(hours=issue_hour)
    )

    heat_series = df.set_index("datetime")["heat_MW"]
    temp_series = df.set_index("datetime")["temperature_2m"]

    # Informacja o cieple dostępna w momencie wydania prognozy
    heat_roll_24h_series = heat_series.rolling(96, min_periods=96).mean()
    heat_roll_48h_series = heat_series.rolling(192, min_periods=192).mean()
    heat_roll_168h_series = heat_series.rolling(672, min_periods=672).mean()

    df["heat_at_issue"] = heat_series.reindex(df["issue_time"]).to_numpy()
    df["heat_roll_mean_24h_at_issue"] = heat_roll_24h_series.reindex(df["issue_time"]).to_numpy()
    df["heat_roll_mean_48h_at_issue"] = heat_roll_48h_series.reindex(df["issue_time"]).to_numpy()
    df["heat_roll_mean_168h_at_issue"] = heat_roll_168h_series.reindex(df["issue_time"]).to_numpy()

    # Bezpieczne lagi ciepła dla całego kolejnego dnia
    df["heat_lag_48h"] = (df["datetime"] - pd.Timedelta(hours=48)).map(heat_series)
    df["heat_lag_168h"] = (df["datetime"] - pd.Timedelta(hours=168)).map(heat_series)

    # Pogoda: dla dnia prognozy traktujemy ją jako prognozę/scenariusz pogody
    df["temp_lag_1h"] = (df["datetime"] - pd.Timedelta(hours=1)).map(temp_series)
    df["temp_lag_3h"] = (df["datetime"] - pd.Timedelta(hours=3)).map(temp_series)
    df["temp_lag_6h"] = (df["datetime"] - pd.Timedelta(hours=6)).map(temp_series)
    df["temp_lag_12h"] = (df["datetime"] - pd.Timedelta(hours=12)).map(temp_series)
    df["temp_lag_24h"] = (df["datetime"] - pd.Timedelta(hours=24)).map(temp_series)
    df["temp_lag_48h"] = (df["datetime"] - pd.Timedelta(hours=48)).map(temp_series)

    temp_roll_3h_series = temp_series.rolling(12, min_periods=12).mean()
    temp_roll_6h_series = temp_series.rolling(24, min_periods=24).mean()
    temp_roll_12h_series = temp_series.rolling(48, min_periods=48).mean()
    temp_roll_24h_series = temp_series.rolling(96, min_periods=96).mean()

    df["temp_roll_mean_3h"] = temp_roll_3h_series.reindex(df["datetime"]).to_numpy()
    df["temp_roll_mean_6h"] = temp_roll_6h_series.reindex(df["datetime"]).to_numpy()
    df["temp_roll_mean_12h"] = temp_roll_12h_series.reindex(df["datetime"]).to_numpy()
    df["temp_roll_mean_24h"] = temp_roll_24h_series.reindex(df["datetime"]).to_numpy()

    return df

## 3. Utworzenie wspólnego pliku ciepła: Helsinki przeskalowane do 15 minut

Ten plik jest wspólną bazą dla treningu i późniejszego użycia w `Serce_modelu`.
Dzięki temu skalowanie nie jest liczone drugi raz w innym miejscu.

In [ ]:
# =========================
# WCZYTANIE SUROWEGO PROFILU HELSINEK I SKALOWANIE RAZ NA STAŁE
# =========================

helsinki_raw = pd.read_csv(
    RAW_HELSINKI_PATH,
    sep=";",
    decimal=",",
)

helsinki_raw["datetime_utc"] = pd.to_datetime(
    helsinki_raw["datetime_utc"],
    format="%d.%m.%Y %H:%M",
)

helsinki_raw = helsinki_raw.rename(
    columns={
        "datetime_utc": "datetime",
        "dh_MWh": "heat_MW_raw",
    }
)

helsinki_raw = helsinki_raw[["datetime", "heat_MW_raw"]].copy()
helsinki_raw = helsinki_raw.sort_values("datetime").reset_index(drop=True)

# Uzupełnienie ewentualnych brakujących godzin przed resamplingiem do 15 minut
full_hourly_range = pd.date_range(
    start=helsinki_raw["datetime"].min(),
    end=helsinki_raw["datetime"].max(),
    freq="h",
)

helsinki_raw = (
    helsinki_raw
    .set_index("datetime")
    .reindex(full_hourly_range)
    .rename_axis("datetime")
    .reset_index()
)

helsinki_raw["heat_MW_raw"] = helsinki_raw["heat_MW_raw"].interpolate(method="linear")

# Globalne skalowanie względem całej historii, a nie lokalnego okna
original_peak_heat_MW_global = helsinki_raw["heat_MW_raw"].max()
scale_factor = THERMAL_POWER_TOTAL_MW / original_peak_heat_MW_global

helsinki_raw["heat_MW"] = helsinki_raw["heat_MW_raw"] * scale_factor

# Resampling do 15 minut
helsinki_scaled_15min = (
    helsinki_raw
    .set_index("datetime")
    .resample("15min")
    .interpolate(method="time")
    .reset_index()
)

# Energia w interwale 15-minutowym
helsinki_scaled_15min["heat_MWh_15min"] = helsinki_scaled_15min["heat_MW"] * 0.25

# Kolumny diagnostyczne — przydadzą się do kontroli spójności skali
helsinki_scaled_15min["scale_factor"] = scale_factor
helsinki_scaled_15min["thermal_power_total_MW"] = THERMAL_POWER_TOTAL_MW
helsinki_scaled_15min["original_peak_heat_MW_global"] = original_peak_heat_MW_global

helsinki_scaled_15min.to_csv(SCALED_HELSINKI_15MIN_PATH, index=False)

print("Zapisano plik:", SCALED_HELSINKI_15MIN_PATH)
print("Zakres dat:", helsinki_scaled_15min["datetime"].min(), "->", helsinki_scaled_15min["datetime"].max())
print("Globalny pik Helsinki przed skalowaniem [MW]:", round(original_peak_heat_MW_global, 2))
print("Docelowy pik po skalowaniu [MW]:", round(THERMAL_POWER_TOTAL_MW, 2))
print("Scale factor:", scale_factor)
print("Liczba wierszy 15-min:", len(helsinki_scaled_15min))

helsinki_scaled_15min.head()

In [ ]:
# Kontrola profilu po skalowaniu
plt.figure(figsize=(14, 5))
plt.plot(helsinki_scaled_15min["datetime"], helsinki_scaled_15min["heat_MW"])
plt.title("Przeskalowane zapotrzebowanie na ciepło — Helsinki -> 5 x J920")
plt.xlabel("Data")
plt.ylabel("Moc cieplna [MW]")
plt.grid(True)
plt.show()

## 4. Pobranie pogody i przygotowanie zbioru modelowego

In [ ]:
# =========================
# WCZYTANIE WSPÓLNEGO PLIKU CIEPŁA 15-MIN
# =========================

data_all_15min = pd.read_csv(SCALED_HELSINKI_15MIN_PATH)
data_all_15min["datetime"] = pd.to_datetime(data_all_15min["datetime"])

data_all_15min = data_all_15min[
    [
        "datetime",
        "heat_MW_raw",
        "heat_MW",
        "heat_MWh_15min",
        "scale_factor",
        "thermal_power_total_MW",
        "original_peak_heat_MW_global",
    ]
].copy()

data_all_15min = data_all_15min.sort_values("datetime").reset_index(drop=True)

print("Zakres danych ciepła:", data_all_15min["datetime"].min(), "->", data_all_15min["datetime"].max())
print("Braki w danych ciepła:")
print(data_all_15min[["heat_MW", "heat_MWh_15min"]].isna().sum())

data_all_15min.head()

In [ ]:
# =========================
# POBRANIE POGODY HISTORYCZNEJ I RESAMPLING DO SIATKI 15-MIN
# =========================

weather_hourly = download_open_meteo_weather(
    LAT,
    LON,
    start_date=str(data_all_15min["datetime"].dt.date.min()),
    end_date=str(data_all_15min["datetime"].dt.date.max()),
)

weather_15min = resample_weather_to_15min_grid(
    weather_hourly=weather_hourly,
    target_datetimes=data_all_15min["datetime"],
)

weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "cloud_cover",
    "shortwave_radiation",
    "precipitation",
]

print("Zakres pogody:", weather_15min["datetime"].min(), "->", weather_15min["datetime"].max())
print("Braki w pogodzie po resamplingu:")
print(weather_15min[weather_cols].isna().sum())

weather_15min.head()

In [ ]:
# =========================
# POŁĄCZENIE CIEPŁA I POGODY
# =========================

data_all_15min = data_all_15min.merge(
    weather_15min,
    on="datetime",
    how="left",
)

if data_all_15min[weather_cols].isna().any().any():
    print(data_all_15min[weather_cols].isna().sum())
    raise ValueError("Są braki w danych pogodowych po merge.")

data_all_15min.to_csv(MODELING_DATA_PATH, index=False)

print("Zapisano zbiór modelowy:", MODELING_DATA_PATH)
print("Wymiary zbioru:", data_all_15min.shape)

data_all_15min.head()

## 5. Budowa cech day-ahead

In [ ]:
# =========================
# BUDOWA CECH DAY-AHEAD
# =========================

data_features = prepare_heat_day_ahead_features(
    df_heat_15min=data_all_15min,
    issue_hour=ISSUE_HOUR,
    base_temp=BASE_TEMP,
)

target = "heat_MW"

features = [
    # Pogoda dla prognozowanego punktu czasu
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "cloud_cover",
    "shortwave_radiation",
    "precipitation",

    # Cechy pogodowe pochodne
    "heating_degree",
    "cooling_degree",

    # Kalendarz
    "month",
    "day",
    "hour",
    "minute",
    "dayofweek",
    "dayofyear",
    "weekofyear",
    "is_weekend",

    # Cykliczność czasu
    "hour_fraction",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "dayofyear_sin",
    "dayofyear_cos",

    # Temperatura historyczna / scenariuszowa
    "temp_lag_1h",
    "temp_lag_3h",
    "temp_lag_6h",
    "temp_lag_12h",
    "temp_lag_24h",
    "temp_lag_48h",
    "temp_roll_mean_3h",
    "temp_roll_mean_6h",
    "temp_roll_mean_12h",
    "temp_roll_mean_24h",

    # Ciepło znane w momencie wydania prognozy D-1 08:00
    "heat_at_issue",
    "heat_roll_mean_24h_at_issue",
    "heat_roll_mean_48h_at_issue",
    "heat_roll_mean_168h_at_issue",

    # Bezpieczne lagi ciepła dla całego kolejnego dnia
    "heat_lag_48h",
    "heat_lag_168h",
]

missing_features = [col for col in features if col not in data_features.columns]
if missing_features:
    raise ValueError(f"Brakuje cech: {missing_features}")

modeling_data = data_features.dropna(subset=features + [target]).copy()

print("Liczba cech:", len(features))
print("Wymiary po usunięciu NaN:", modeling_data.shape)
print("Zakres modelowy:", modeling_data["datetime"].min(), "->", modeling_data["datetime"].max())

modeling_data[["datetime", target] + features[:8]].head()

## 6. Podział train/test/reference

Zgodnie z założeniem:
- trening: lata `<= 2022`,
- test właściwy: `2023`,
- rok referencyjny/scenariuszowy: `2024`.

In [ ]:
# =========================
# PODZIAŁ TRAIN / TEST / REFERENCE
# =========================

train = modeling_data[modeling_data["year"] <= 2022].copy()
test_2023 = modeling_data[modeling_data["year"] == 2023].copy()
reference_2024 = modeling_data[modeling_data["year"] == 2024].copy()

X_train = train[features].copy()
y_train = train[target].copy()

X_test_2023 = test_2023[features].copy()
y_test_2023 = test_2023[target].copy()

X_reference_2024 = reference_2024[features].copy()
y_reference_2024 = reference_2024[target].copy()

print("Train <=2022:", X_train.shape)
print("Test 2023:", X_test_2023.shape)
print("Reference 2024:", X_reference_2024.shape)
print("Target:", target)

## 7. Trening modelu day-ahead

In [ ]:
# =========================
# TRENING MODELU DAY-AHEAD
# =========================

model_day_ahead = XGBRegressor(
    n_estimators=900,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
)

model_day_ahead.fit(X_train, y_train)

pred_train = np.maximum(model_day_ahead.predict(X_train), 0)
pred_test_2023 = np.maximum(model_day_ahead.predict(X_test_2023), 0)
pred_reference_2024 = np.maximum(model_day_ahead.predict(X_reference_2024), 0)

metrics_train = calculate_metrics(y_train, pred_train)
metrics_2023 = calculate_metrics(y_test_2023, pred_test_2023)
metrics_2024 = calculate_metrics(y_reference_2024, pred_reference_2024)

summary_metrics = pd.DataFrame([
    {"zbior": "train", **metrics_train},
    {"zbior": "test_2023", **metrics_2023},
    {"zbior": "reference_2024", **metrics_2024},
])

summary_metrics

In [ ]:
# =========================
# WYNIKI DLA TESTU I ROKU REFERENCYJNEGO
# =========================

test_2023_results = test_2023[["datetime", "heat_MW", "heat_MWh_15min"]].copy()
test_2023_results["prediction_MW"] = pred_test_2023
test_2023_results["prediction_MWh_15min"] = test_2023_results["prediction_MW"] * 0.25
test_2023_results["error_MW"] = test_2023_results["prediction_MW"] - test_2023_results["heat_MW"]

reference_2024_results = reference_2024[["datetime", "heat_MW", "heat_MWh_15min"]].copy()
reference_2024_results["prediction_MW"] = pred_reference_2024
reference_2024_results["prediction_MWh_15min"] = reference_2024_results["prediction_MW"] * 0.25
reference_2024_results["error_MW"] = reference_2024_results["prediction_MW"] - reference_2024_results["heat_MW"]

print("Suma rzeczywista 2023 [MWh]:", round(test_2023_results["heat_MWh_15min"].sum(), 2))
print("Suma prognozowana 2023 [MWh]:", round(test_2023_results["prediction_MWh_15min"].sum(), 2))
print("Suma rzeczywista 2024 [MWh]:", round(reference_2024_results["heat_MWh_15min"].sum(), 2))
print("Suma prognozowana 2024 [MWh]:", round(reference_2024_results["prediction_MWh_15min"].sum(), 2))

In [ ]:
# =========================
# WAŻNOŚĆ CECH
# =========================

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model_day_ahead.feature_importances_,
}).sort_values("importance", ascending=False)

display(feature_importance.head(30))

plt.figure(figsize=(10, 8))
plt.barh(
    feature_importance.head(25)["feature"][::-1],
    feature_importance.head(25)["importance"][::-1],
)
plt.title("Najważniejsze cechy modelu day-ahead")
plt.xlabel("Importance")
plt.ylabel("Cecha")
plt.grid(True)
plt.show()

## 9. Zapis modelu do pliku `.pkl`

Zapisujemy model razem z informacją o skali i nazwą wspólnego pliku 15-minutowego. Ten plik powinien być potem używany w `Serce_modelu`.

In [ ]:
# =========================
# ZAPIS MODELU DAY-AHEAD
# =========================

heat_bundle = {
    "model": model_day_ahead,
    "features": features,
    "target": target,
    "issue_hour": ISSUE_HOUR,
    "base_temp": BASE_TEMP,
    "unit_target": "MW",
    "scaled_heat_file": SCALED_HELSINKI_15MIN_PATH,
    "modeling_data_file": MODELING_DATA_PATH,
    "scale_factor": scale_factor,
    "thermal_power_total_MW": THERMAL_POWER_TOTAL_MW,
    "original_peak_heat_MW_global": original_peak_heat_MW_global,
    "train_period": "<=2022",
    "test_period": "2023",
    "reference_period": "2024",
    "description": (
        "Model day-ahead zapotrzebowania na ciepło. "
        "Prognoza D-1 08:00 na cały dzień D. "
        "Dane wejściowe: Helsinki przeskalowane raz do 15 minut."
    ),
}

joblib.dump(heat_bundle, HEAT_MODEL_PATH)

print("Zapisano model:", HEAT_MODEL_PATH)
print("Plik ciepła użyty do treningu:", SCALED_HELSINKI_15MIN_PATH)
print("Zbiór modelowy:", MODELING_DATA_PATH)
print("Target:", target)
print("Jednostka targetu:", heat_bundle["unit_target"])
print("Liczba cech:", len(features))
print("Scale factor:", scale_factor)

In [ ]:
# =========================
# SZYBKI TEST WCZYTANIA MODELU
# =========================

loaded_bundle = joblib.load(HEAT_MODEL_PATH)

print("Opis:", loaded_bundle.get("description"))
print("Target:", loaded_bundle.get("target"))
print("Jednostka:", loaded_bundle.get("unit_target"))
print("Issue hour:", loaded_bundle.get("issue_hour"))
print("Scaled heat file:", loaded_bundle.get("scaled_heat_file"))
print("Scale factor:", loaded_bundle.get("scale_factor"))
print("Liczba cech:", len(loaded_bundle.get("features", [])))